# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [3]:
# Setup — same pattern as w03: getpass, never pasted, REL/TABLES as before
from getpass import getpass
import duckdb, os
import pandas as pd


HF_TOKEN = os.environ.get('HF_TOKEN') or getpass('Paste your HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
tbl = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)"

raw = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS max_d FROM {tbl}),
    decision AS (SELECT max_d - INTERVAL 30 DAY AS decision_date FROM bounds),
    prior AS (
        SELECT f.content_hash_id,
            AVG(f.gsc_clicks)                                                 AS avg_daily_clicks_prior,
            AVG(f.gsc_avg_position) FILTER (WHERE f.gsc_avg_position > 0)     AS avg_position_prior,
            COUNT(*) FILTER (WHERE f.gsc_clicks > 0)                          AS days_with_clicks_prior,
            AVG(CASE WHEN f.ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) AS ga4_coverage_prior,
            SUM(f.sessions_ai) * 1.0 / NULLIF(SUM(f.sessions_organic + f.sessions_ai), 0) AS ai_share_prior,
            COUNT(*) AS n_days_prior
        FROM {tbl} f, decision d
        WHERE f.report_date >= d.decision_date - INTERVAL 90 DAY
          AND f.report_date <  d.decision_date
        GROUP BY f.content_hash_id
        HAVING COUNT(*) >= 30
    ),
    future AS (
        SELECT f.content_hash_id, AVG(f.gsc_clicks) AS avg_daily_clicks_future
        FROM {tbl} f, decision d
        WHERE f.report_date >= d.decision_date AND f.report_date < d.decision_date + INTERVAL 30 DAY
        GROUP BY f.content_hash_id
    )
    SELECT p.*, fu.avg_daily_clicks_future
    FROM prior p JOIN future fu USING (content_hash_id)
""").df()

print(f'{len(raw):,} content items with full prior+future coverage')

# --- Missing-value handling (deliberate, not fillna(0) everywhere) ---
raw['no_ranking_data_prior'] = raw['avg_position_prior'].isna().astype(int)
raw['avg_position_prior'] = raw['avg_position_prior'].fillna(100)   # sentinel: worse than any real rank, never "best"
raw['ai_share_prior'] = raw['ai_share_prior'].fillna(0)             # zero organic+AI sessions -> treat AI share as 0

# --- Engineered features ---
raw['click_consistency_prior'] = raw['days_with_clicks_prior'] / raw['n_days_prior']  # normalizes window-length noise

# --- Categorical handling: position bucket, one-hot encoded ---
raw['position_bucket'] = pd.cut(
    raw['avg_position_prior'], bins=[-1, 10, 20, 50, 100],
    labels=['top10', 'p11_20', 'p21_50', 'beyond_50_or_unranked']
)
bucket_dummies = pd.get_dummies(raw['position_bucket'], prefix='pos')

feature_cols = ['avg_daily_clicks_prior', 'avg_position_prior', 'click_consistency_prior',
                 'ga4_coverage_prior', 'ai_share_prior', 'no_ranking_data_prior']
X = pd.concat([raw[feature_cols], bucket_dummies], axis=1)

raw['is_declining_future'] = (
    (raw['avg_daily_clicks_future'] < 0.75 * raw['avg_daily_clicks_prior']) &
    (raw['avg_daily_clicks_prior'] >= 1.0)
).astype(int)
y = raw['is_declining_future']

print('base rate:', y.mean())
X.head()

Paste your HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

363,626 content items with full prior+future coverage
base rate: 0.012686111554179294


,avg_daily_clicks_prior,avg_position_prior,click_consistency_prior,ga4_coverage_prior,ai_share_prior,no_ranking_data_prior,pos_top10,pos_p11_20,pos_p21_50,pos_beyond_50_or_unranked
0,0.0,100.000000,0.0,0.0,0.0,1,False,False,False,True
1,0.0,14.666667,0.0,0.0,0.0,0,False,True,False,False
2,0.0,100.000000,0.0,0.0,0.0,1,False,False,False,True
3,0.0,5.000000,0.0,0.0,0.0,0,True,False,False,False
4,0.0,100.000000,0.0,0.0,0.0,1,False,False,False,True


In [4]:
import pandas as pd
notes = pd.DataFrame([
    {'feature': 'avg_daily_clicks_prior', 'meaning': 'mean daily GSC clicks, prior 90d', 'missing': 'none (count-filtered)', 'available_before_decision': True},
    {'feature': 'avg_position_prior', 'meaning': 'mean search position, prior 90d, ranked days only', 'missing': f"filled 100 (sentinel) + no_ranking_data_prior flag, {raw['no_ranking_data_prior'].sum():,} rows", 'available_before_decision': True},
    {'feature': 'click_consistency_prior', 'meaning': 'share of prior-window days with any clicks', 'missing': 'none', 'available_before_decision': True},
    {'feature': 'ga4_coverage_prior', 'meaning': 'share of prior-window days GA4 data is trustworthy', 'missing': 'none', 'available_before_decision': True},
    {'feature': 'ai_share_prior', 'meaning': 'share of organic+AI sessions from AI surfaces', 'missing': f"filled 0, {raw['ai_share_prior'].isna().sum():,} rows had zero denominator", 'available_before_decision': True},
    {'feature': 'position_bucket (one-hot)', 'meaning': 'categorical rank tier from avg_position_prior', 'missing': 'n/a (derived post-fill)', 'available_before_decision': True},
])
notes

,feature,meaning,missing,available_before_decision
0,avg_daily_clicks_prior,"mean daily GSC clicks, prior 90d",none (count-filtered),True
1,avg_position_prior,"mean search position, prior 90d, ranked days only",filled 100 (sentinel) + no_ranking_data_prior ...,True
2,click_consistency_prior,share of prior-window days with any clicks,none,True
3,ga4_coverage_prior,share of prior-window days GA4 data is trustwo...,none,True
4,ai_share_prior,share of organic+AI sessions from AI surfaces,"filled 0, 0 rows had zero denominator",True
5,position_bucket (one-hot),categorical rank tier from avg_position_prior,n/a (derived post-fill),True


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
honest_auc = roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])
print('Honest AUC:', honest_auc)

# Attack: smuggle the future-window value in directly
Xl = X.copy()
Xl['avg_daily_clicks_future'] = raw['avg_daily_clicks_future']
Xtrl, Xtel, _, _ = train_test_split(Xl, y, test_size=0.3, random_state=0, stratify=y)
leaky_auc = roc_auc_score(yte, LogisticRegression(max_iter=1000).fit(Xtrl, ytr).predict_proba(Xtel)[:, 1])
print('Leaky AUC (should jump toward 1.0):', leaky_auc)

# Correlation check: any real feature suspiciously tied to the label?
print(X.assign(label=y).corr()['label'].sort_values(ascending=False))

Honest AUC: 0.9949640196165985
Leaky AUC (should jump toward 1.0): 0.9995243063948762
label                        1.000000
click_consistency_prior      0.602058
avg_daily_clicks_prior       0.425197
ga4_coverage_prior           0.368675
pos_top10                    0.121205
pos_p11_20                   0.012038
ai_share_prior              -0.005707
pos_p21_50                  -0.025212
no_ranking_data_prior       -0.087915
pos_beyond_50_or_unranked   -0.096998
avg_position_prior          -0.111627
Name: label, dtype: float64


In [6]:
eligible = raw['avg_daily_clicks_prior'] >= 1.0
X_e, y_e = X[eligible], y[eligible]
print(f'eligible population: {eligible.sum():,} of {len(raw):,} ({eligible.mean():.1%})')
print('base rate within eligible population:', y_e.mean())

Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(X_e, y_e, test_size=0.3, random_state=0, stratify=y_e)
clf_e = LogisticRegression(max_iter=1000).fit(Xtr_e, ytr_e)
honest_auc_eligible = roc_auc_score(yte_e, clf_e.predict_proba(Xte_e)[:, 1])
print('Honest AUC, eligible-only:', honest_auc_eligible)

eligible population: 6,920 of 363,626 (1.9%)
base rate within eligible population: 0.6666184971098266
Honest AUC, eligible-only: 0.6629356142871463


In [7]:
print(raw['avg_daily_clicks_prior'].describe())
print()
print('percentiles:', raw['avg_daily_clicks_prior'].quantile([.5, .9, .95, .98, .99, .999]).to_dict())
print()
print(f"eligible (>=1 click/day avg): {eligible.sum():,} of {len(raw):,} ({eligible.mean():.1%})")

count    363626.000000
mean          0.081198
std           0.624759
min           0.000000
25%           0.000000
50%           0.000000
75%           0.011111
max         190.877778
Name: avg_daily_clicks_prior, dtype: float64

percentiles: {0.5: 0.0, 0.9: 0.1111111111111111, 0.95: 0.3387096774193548, 0.98: 0.9444444444444444, 0.99: 1.4985955056179776, 0.999: 6.377777777777778}

eligible (>=1 click/day avg): 6,920 of 363,626 (1.9%)


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [8]:
def signal_test(name, series_pos, series_neg):
    diff = series_pos.mean() - series_neg.mean()
    print(f"{name}: decline-group mean={series_pos.mean():.3f}  no-decline mean={series_neg.mean():.3f}  diff={diff:+.3f}")
    return diff

pos = raw.loc[eligible & (y_e == 1)]
neg = raw.loc[eligible & (y_e == 0)]

print("--- Signal 1: weak position -> decline ---")
d1 = signal_test('avg_position_prior', pos['avg_position_prior'], neg['avg_position_prior'])
# verdict: CONFIRMED if d1 clearly positive (declining pages ranked worse), OPPOSITE if negative, MIXED if near zero

print("\n--- Signal 2: click consistency -> decline ---")
d2 = signal_test('click_consistency_prior', pos['click_consistency_prior'], neg['click_consistency_prior'])

print("\n--- Signal 3: GA4 coverage -> decline ---")
d3 = signal_test('ga4_coverage_prior', pos['ga4_coverage_prior'], neg['ga4_coverage_prior'])

--- Signal 1: weak position -> decline ---
avg_position_prior: decline-group mean=8.938  no-decline mean=7.374  diff=+1.564

--- Signal 2: click consistency -> decline ---
click_consistency_prior: decline-group mean=0.708  no-decline mean=0.747  diff=-0.040

--- Signal 3: GA4 coverage -> decline ---
ga4_coverage_prior: decline-group mean=0.470  no-decline mean=0.609  diff=-0.139


In [9]:
# does GA4 coverage look like a per-page signal, or a per-client constant?
raw_with_client = con.sql(f"""
    SELECT DISTINCT content_hash_id, client_hash_id FROM {tbl}
""").df()
check = raw.merge(raw_with_client, on='content_hash_id', how='left')
per_client_var = check.groupby('client_hash_id')['ga4_coverage_prior'].std()
print('median within-client std dev of ga4_coverage_prior:', per_client_var.median())
print('overall std dev:', check['ga4_coverage_prior'].std())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

median within-client std dev of ga4_coverage_prior: 0.03649729840440311
overall std dev: 0.12798276927524746


In [11]:
weak_position = raw.loc[eligible, 'avg_position_prior'] > raw.loc[eligible, 'avg_position_prior'].median()
decline_rate_weak = y_e[weak_position.values].mean()
decline_rate_strong = y_e[~weak_position.values].mean()
print(f"decline rate, weak-position half: {decline_rate_weak:.3f}")
print(f"decline rate, strong-position half: {decline_rate_strong:.3f}")
print(f"base rate (all eligible): {y_e.mean():.3f}")

decline rate, weak-position half: 0.731
decline rate, strong-position half: 0.603
base rate (all eligible): 0.667


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Flag tested: "weak search position" (the flag most FlyRank content-review rules key off).
Among eligible pages (>=1 avg daily click, prior 90d), split at the median avg_position_prior:
  - weak-position half: 73.1% declined
  - strong-position half: 60.3% declined
  - base rate: 66.7%

Verdict: CONFIRMED. Weak position doesn't just correlate directionally (Signal 1) — splitting the
eligible population at the median produces a 13-point spread in actual decline rate, wide enough
to be operationally useful for a reviewer's rule, not just statistically present.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

For a content team: search position is a legitimate early-warning signal for decline, but only
among the ~1.9% of pages that already clear a meaningful traffic floor — position tells you almost
nothing about the other 98.1%, because those pages don't have enough volume for "decline" to be a
coherent question in the first place. Click consistency and GA4 coverage looked promising at first
glance but didn't hold up under scrutiny: consistency's effect was too thin to trust, and GA4
coverage turned out to mostly reflect which client set up tracking properly, not page-level
behavior — a reviewer using it as a decline flag would really be reacting to account hygiene.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.